---
title: "Vector data, hands-on"
subtitle: "GeoParquet, FlatGeobuf, PMTiles, and an embeddings bonus, all on the same Salzburg AOI"
date: today
format:
  html:
    toc: true
    code-fold: false
    code-tools: true
    code-line-numbers: true
    code-overflow: wrap
    code-copy: true
    df-print: paged
    link-external-icon: true
    link-external-newwindow: true
    smooth-scroll: true
    embed-resources: true
    html-table-processing: none
execute:
  echo: true
  warning: false
  cache: false
  freeze: false
jupyter: python3
---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshitijrajsharma/cng-workshop-materials/blob/master/notebooks/01_vector.ipynb)
[![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-F37626?logo=jupyter&logoColor=white)](https://github.com/kshitijrajsharma/cng-workshop-materials/raw/master/notebooks/01_vector.ipynb)
[![Repo](https://img.shields.io/badge/Source-GitHub-181717?logo=github&logoColor=white)](https://github.com/kshitijrajsharma/cng-workshop-materials)

**What you'll do.** Pull POIs from **Overture**'s planet-scale GeoParquet on S3 with a single SQL query, then read three different cloud-native vector formats of the same Austria buildings dataset (**GeoParquet**, **FlatGeobuf**, **PMTiles**) directly off Hugging Face over HTTPS, fetching only the bytes that intersect Salzburg. We visualise everything on a `lonboard` map. Finish with a bonus: 64-D **Alpha Earth embeddings** opened as Zarr, clustered into land-cover groups.

In [ ]:
# | eval: false
# Uncomment and run this cell once if you are on Colab or a fresh kernel.
# !pip install duckdb geopandas pyogrio lonboard shapely pmtiles pyarrow zarr obstore scipy fsspec  # noqa: E501

In [ ]:
import json
import urllib.request

import duckdb
import fsspec
import geopandas as gpd
import pyarrow.parquet as pq
import pyogrio
from lonboard import Map, PolygonLayer, ScatterplotLayer
from pmtiles.reader import Reader

REPO_RAW = "https://raw.githubusercontent.com/kshitijrajsharma/cng-workshop-materials/master"
AOI_URL = f"{REPO_RAW}/data/aoi/salzburg_bbox.json"
BBOX = tuple(json.loads(urllib.request.urlopen(AOI_URL).read())["bbox"])
xmin, ymin, xmax, ymax = BBOX

BASE = "https://huggingface.co/datasets/kshitijrajsharma/cng-workshop-materials/resolve/main"

AUSTRIA_GEOPARQUET = f"{BASE}/austria_buildings.geo.parquet"
AUSTRIA_FGB = f"{BASE}/austria_buildings.fgb"
AUSTRIA_PMTILES = f"{BASE}/austria_buildings.pmtiles"
print(f"AOI bbox = {BBOX}")

## 1. Overture, query a planet-scale GeoParquet on S3 with SQL

Overture publishes the planet as partitioned GeoParquet on a public S3 bucket. The S3 layout is **Hive-partitioned**: each combination of theme + type lives in its own subfolder (`theme=places/type=place/...`). With `hive_partitioning=1`, DuckDB reads that structure as columns and prunes whole subtrees before touching any data. Inside the surviving parquet files, the GeoParquet `bbox` struct lets DuckDB push the spatial predicate down to **row-group filtering**: only the row groups whose extent intersects Salzburg are fetched, the rest of the planet is skipped at the byte level.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

OVERTURE = "s3://overturemaps-us-west-2/release/2026-04-15.0/theme=places/type=place/*"
places_df = con.execute(
    f"""
    SELECT id, names.primary AS name, categories.primary AS category, ST_AsText(geometry) AS wkt
    FROM read_parquet('{OVERTURE}', hive_partitioning=1)
    WHERE bbox.xmin BETWEEN {xmin} AND {xmax}
      AND bbox.ymin BETWEEN {ymin} AND {ymax}
    """
).fetchdf()
places = gpd.GeoDataFrame(
    places_df.drop(columns=["wkt"]),
    geometry=gpd.GeoSeries.from_wkt(places_df["wkt"]),
    crs="EPSG:4326",
)
print(f"Overture places inside Salzburg AOI = {len(places):,}")
places.head(3)

Drop the result onto a map.

In [ ]:
Map(
    [
        ScatterplotLayer.from_geopandas(
            places, get_fill_color=[40, 160, 80, 200], radius_min_pixels=3
        )
    ]
)

## 2. GeoParquet, read the metadata first, then load directly on a map

Parquet stores rows in **row groups** (typically 100k-1M rows each), each compressed independently, with per-column statistics in the file footer. **GeoParquet 1.1** adds a `bbox` covering column whose statistics describe the spatial extent of each row group. A reader can therefore look at the footer (a few KB), decide which row groups intersect the query bbox, and fetch only those. Inspect the metadata first; feature data is fetched only once we know what we want.

In [ ]:
fs, path = fsspec.url_to_fs(AUSTRIA_GEOPARQUET)
with fs.open(path, "rb") as f:
    pf = pq.ParquetFile(f)
    print(f"file size      = {fs.size(path) / 1e6:.1f} MB")
    print(f"rows           = {pf.metadata.num_rows:,}")
    print(f"row groups     = {pf.metadata.num_row_groups}")
    print(f"schema columns = {pf.schema_arrow.names}")
    geo = json.loads(pf.schema_arrow.metadata[b"geo"])
print("geo metadata:")
print(json.dumps(geo, indent=2))

`gpd.read_parquet(file, bbox=...)` uses the bbox covering column to skip row groups that do not intersect. From a 1 GB+ file on HF we pull only the Salzburg row groups.

In [ ]:
with fs.open(path, "rb") as f:
    buildings_gp = gpd.read_parquet(f, bbox=BBOX)
print(f"Salzburg buildings (GeoParquet, bbox pushdown) = {len(buildings_gp):,}")

lonboard renders the GeoDataFrame directly via the GeoArrow-backed deck.gl pipeline

In [ ]:
Map([PolygonLayer.from_geopandas(buildings_gp, get_fill_color=[200, 80, 80, 160])])

## 3. FlatGeobuf, header + R-tree, bbox-stream over HTTP

A FlatGeobuf file lays out three parts back-to-back: a binary **header** (CRS, feature count, geometry type, columns), a **packed Hilbert R-tree** index, then the features in spatial order. The R-tree lets a reader resolve "which feature blocks intersect this bbox?" with a handful of HTTP range requests, before fetching the matching feature bytes. `pyogrio.read_info(url)` reads only the header.

In [ ]:
info = pyogrio.read_info(AUSTRIA_FGB)
print(f"driver         = {info['driver']}")
print(f"feature count  = {info['features']:,}")
print(f"geometry type  = {info['geometry_type']}")
print(f"crs            = {info['crs']}")
print(f"fields         = {list(info['fields'])}")

`bbox=` triggers an R-tree partial read, only the feature blocks intersecting the bbox are fetched. Same code path for local paths and `https://...`.

In [ ]:
buildings_fgb = pyogrio.read_dataframe(AUSTRIA_FGB, bbox=BBOX)
print(f"Salzburg buildings (FGB, R-tree pushdown) = {len(buildings_fgb):,}")

Same Salzburg subset, two formats, two spatial indexes. Drop both onto one map alongside the Overture places.

In [ ]:
Map(
    [
        PolygonLayer.from_geopandas(buildings_fgb, get_fill_color=[80, 80, 200, 160]),
        ScatterplotLayer.from_geopandas(
            places,
            get_fill_color=[40, 160, 80, 255],
            radius_min_pixels=3,
        ),
    ]
)

## 4. PMTiles, one file, browser-side tile streaming

Classic web maps need a tile server (a process that takes `z/x/y` requests and returns tile bytes). PMTiles collapses that into a single file: every tile of the pyramid plus a `(z, x, y) → byte offset` directory. A browser viewer fetches the 16-byte header, then issues HTTP range requests for just the tiles in the current viewport. The entire stack becomes a static file on object storage plus a viewer.


In [ ]:
def pmtiles_get_bytes(url_or_path):
    f = fsspec.open(str(url_or_path), "rb").open()

    def get(offset, length):
        f.seek(offset)
        return f.read(length)

    return get


header = Reader(pmtiles_get_bytes(AUSTRIA_PMTILES)).header()
print(f"min/max zoom   = {header['min_zoom']} .. {header['max_zoom']}")
print(f"tile type      = {header['tile_type'].name}")
print(f"tile compress  = {header['tile_compression'].name}")
print(
    f"bounds         = ({header['min_lon_e7'] / 1e7}, {header['min_lat_e7'] / 1e7}, "
    f"{header['max_lon_e7'] / 1e7}, {header['max_lat_e7'] / 1e7})"
)

Open the whole Austria buildings in the official viewer, the browser streams just the tiles in view:

<https://pmtiles.io/?url=https://huggingface.co/datasets/kshitijrajsharma/cng-workshop-materials/resolve/main/austria_buildings.pmtiles>

## Where each format shines

| Format       | Indexing                        | Best for                                                 |
| ------------ | ------------------------------- | -------------------------------------------------------- |
| GeoParquet   | Bbox covering column, row-group pruning | Column-aware analytics, DataFrame workflows, lonboard    |
| FlatGeobuf   | Packed R-tree at file head      | Bbox feature reads, GIS tools (QGIS, ogr2ogr, JS readers) |
| PMTiles      | Z/X/Y directory + tile offsets  | Browser map rendering at any zoom, served straight from object storage |

## Bonus: the same trick on ML embeddings

An **embedding** is a learned vector that summarises what's in a pixel (or feature). Two pixels with similar land cover get similar vectors, so distance in vector space approximates similarity on the ground. [Alpha Earth Foundations](https://geoembeddings.org/tutorials/xarray_geospatial_embeddings_intro.html) publishes 64-dimensional embeddings for every 10 m Sentinel-2 pixel on Earth, packaged as a public **Zarr v3** on Source Cooperative S3. Physical shards of `(1, 64, 4096, 4096)` (~1 GiB raw int8 each) wrap a grid of `(1, 64, 256, 256)` logical chunks (~4 MiB), so a small AOI never pulls the whole shard.

Open the store with `obstore` (direct S3 byte ranges) and let `zarr-python` do the chunk-level reads, exactly as the tutorial does. `read_aoi` wraps the bbox-to-pixel-index math through the affine transform on the group attrs; `y` is descending (north to south), hence `ymax` maps to the top of the slice. We stride by 4 (~40 m) to keep the array small for Colab. `pixel_layer` is the lonboard layer we will reuse on every map.

In [ ]:
import math

import numpy as np
import zarr
from obstore.store import S3Store
from scipy.cluster.vq import kmeans2, vq
from zarr.storage import ObjectStore

AEF = "s3://us-west-2.opendata.source.coop/tge-labs/aef-mosaic/"
store = S3Store.from_url(AEF, skip_signature=True, region="us-west-2")
g = zarr.open_group(store=ObjectStore(store, read_only=True), mode="r")
emb = g["embeddings"]
dx, _, x_origin, _, dy, y_origin = g.attrs["spatial:transform"]


def read_aoi(bbox, stride=4):
    """bbox -> (flat (n, 64) float32 embeddings, flat lon, flat lat) for the most recent year."""
    xmn, ymn, xmx, ymx = bbox
    x0 = math.floor((xmn - x_origin) / dx)
    x1 = math.ceil((xmx - x_origin) / dx)
    y0 = math.floor((ymx - y_origin) / dy)
    y1 = math.ceil((ymn - y_origin) / dy)
    block = emb[-1, :, y0:y1:stride, x0:x1:stride]
    flat = block.transpose(1, 2, 0).astype(np.float32).reshape(-1, 64)
    xs = x_origin + (x0 + np.arange(0, x1 - x0, stride)) * dx
    ys = y_origin + (y0 + np.arange(0, y1 - y0, stride)) * dy
    yy, xx = np.meshgrid(ys, xs, indexing="ij")
    return flat, xx.ravel(), yy.ravel()


def pixel_layer(lon, lat, labels, palette, radius=1):
    return ScatterplotLayer.from_geopandas(
        gpd.GeoDataFrame(geometry=gpd.points_from_xy(lon, lat), crs="EPSG:4326"),
        get_fill_color=palette[labels],
        radius_min_pixels=radius,
        pickable=False,
        auto_highlight=False,
    )


print(f"shape  = {emb.shape}   dtype = {emb.dtype}")
print(f"chunks = {emb.chunks}   shards = {emb.shards}")

Cluster the Salzburg per-pixel embedding vectors into 5 groups and colour by cluster on lonboard.

In [ ]:
flat, slon, slat = read_aoi(BBOX)
_, labels = kmeans2(flat, 5, seed=0, minit="++")
palette = np.array(
    [
        [230, 25, 75],
        [60, 180, 75],
        [255, 195, 0],
        [0, 130, 200],
        [245, 130, 48],
    ],
    dtype=np.uint8,
)
print(f"Salzburg pixels = {len(flat):,}  ({flat.nbytes / 1e6:.2f} MB)")
Map([pixel_layer(slon, slat, labels, palette)])

### Training GeoAI model zarr embeedings cube

The k-means above is unsupervised: it finds five groups but does not name them. To put `vegetation` or `water` labels on the output we hand-pick a couple of example pixels per class, average their 64-D embeddings into one prototype per class, and reuse `vq` for inference. The whole model is `(n_classes, 64)` floats, ~200 numbers for 3 classes.

In [ ]:
TRAINING = {
    "water": [(13.044, 47.800), (13.044, 47.795)],  # Salzach river
    "vegetation": [(13.058, 47.803), (13.063, 47.825)],  # Kapuzinerberg, northern forest
    "urban": [(13.046, 47.813), (13.046, 47.800)],  # Hauptbahnhof, old town
}


def emb_at(lon, lat):
    xi = math.floor((lon - x_origin) / dx)
    yi = math.floor((lat - y_origin) / dy)
    return emb[-1, :, yi, xi].astype(np.float32)


classes = list(TRAINING)
prototypes = np.array(
    [np.mean([emb_at(lon, lat) for lon, lat in pts], axis=0) for pts in TRAINING.values()]
)
class_colors = np.array(
    [[0, 130, 200], [60, 180, 75], [230, 25, 75]],  # water, vegetation, urban
    dtype=np.uint8,
)
print(f"prototypes = {prototypes.shape} ({prototypes.size} params)")

The model is a **nearest class prototype** classifier (aka nearest centroid): one 64-D mean per class, 192 numbers for 3 classes.

In [ ]:
s_idx, _ = vq(flat, prototypes)
print({c: f"{100 * (s_idx == j).mean():.1f}%" for j, c in enumerate(classes)})

train_lon = [lon for pts in TRAINING.values() for lon, _ in pts]
train_lat = [lat for pts in TRAINING.values() for _, lat in pts]
train_cls = np.array([j for j, pts in enumerate(TRAINING.values()) for _ in pts])

Map(
    [
        pixel_layer(slon, slat, s_idx, class_colors),
        ScatterplotLayer.from_geopandas(
            gpd.GeoDataFrame(geometry=gpd.points_from_xy(train_lon, train_lat), crs="EPSG:4326"),
            get_fill_color=class_colors[train_cls],
            stroked=True,
            get_line_color=[0, 0, 0],
            line_width_min_pixels=2,
            radius_min_pixels=8,
        ),
    ]
)

Apply the same prototypes to Innsbruck.

In [ ]:
i_flat, ilon, ilat = read_aoi((11.34, 47.24, 11.46, 47.30))
i_idx, _ = vq(i_flat, prototypes)
print({c: f"{100 * (i_idx == j).mean():.1f}%" for j, c in enumerate(classes)})
Map([pixel_layer(ilon, ilat, i_idx, class_colors)])

## References

- Overture + DuckDB: <https://docs.overturemaps.org/getting-data/duckdb/>
- GeoParquet 1.1: <https://geoparquet.org/releases/v1.1.0/>
- FlatGeobuf: <https://flatgeobuf.org/>
- lonboard: <https://developmentseed.org/lonboard/>
- PMTiles: <https://docs.protomaps.com/pmtiles/>